In [3]:
import spacy
import pandas as pd

In [4]:
df = pd.read_csv("../data/expense_ner.csv")
df

,text,item,price,currency,merchant,category,date_ref
0,I went to the mall today and bought a T-shirt ...,T-shirt,10.00,USD,mall,Clothing,today
1,Paid 150 EGP for an Uber ride to Cairo Festiva...,Uber ride,150.00,EGP,Uber,Transport,NaN
2,Grabbed a large iced latte at Starbucks for $5...,large iced latte,5.75,USD,Starbucks,Food & Drink,this morning
3,Ordered a pepperoni pizza from Papa Johns for ...,pepperoni pizza,320.00,EGP,Papa Johns,Food & Drink,NaN
4,Bought wireless noise cancelling headphones on...,wireless noise cancelling headphones,199.99,USD,Amazon,Electronics,NaN
...,...,...,...,...,...,...,...
195,I went to Starbucks and bought a laptop stand ...,laptop stand,7.51,USD,Starbucks,Electronics,NaN
196,Ordered a laptop stand from Tradeline for 1790...,laptop stand,1790.00,EGP,Tradeline,Electronics,last night
197,I went to Walmart and bought wireless earbuds ...,wireless earbuds,121.78,USD,Walmart,Electronics,NaN
198,Spent 3000.0 EGP on sunglasses at Tradeline today,sunglasses,3000.00,EGP,Tradeline,Accessories,today


In [6]:
df = df.fillna("")
df

,text,item,price,currency,merchant,category,date_ref
0,I went to the mall today and bought a T-shirt ...,T-shirt,10.00,USD,mall,Clothing,today
1,Paid 150 EGP for an Uber ride to Cairo Festiva...,Uber ride,150.00,EGP,Uber,Transport,
2,Grabbed a large iced latte at Starbucks for $5...,large iced latte,5.75,USD,Starbucks,Food & Drink,this morning
3,Ordered a pepperoni pizza from Papa Johns for ...,pepperoni pizza,320.00,EGP,Papa Johns,Food & Drink,
4,Bought wireless noise cancelling headphones on...,wireless noise cancelling headphones,199.99,USD,Amazon,Electronics,
...,...,...,...,...,...,...,...
195,I went to Starbucks and bought a laptop stand ...,laptop stand,7.51,USD,Starbucks,Electronics,
196,Ordered a laptop stand from Tradeline for 1790...,laptop stand,1790.00,EGP,Tradeline,Electronics,last night
197,I went to Walmart and bought wireless earbuds ...,wireless earbuds,121.78,USD,Walmart,Electronics,
198,Spent 3000.0 EGP on sunglasses at Tradeline today,sunglasses,3000.00,EGP,Tradeline,Accessories,today


In [11]:
nlp = spacy.load("en_core_web_md")

In [45]:
doc = df["text"][6]
doc

'Filled up the gas tank at Mobil for 450 EGP'

In [52]:
def posPrint(doc):
    text = nlp(doc)
    for token in text:
        if not token.is_space:
            print(f"{token.text:<12} POS: {token.pos_:<6} Fine: {token.tag_}")

def itemsBoughtFromNouns(doc):
    text = nlp(doc)
    i = 0
    items = []
    for token in text:
        if not token.is_space and token.pos_ == "NOUN" and i != 2:
            # print(f"{token.text:<12} POS: {token.pos_:<6} Fine: {token.tag_}")
            items.append(token.text)
            i += 1
    print(doc, " - Items Bought:", items)

def nerPrint(doc):
    text = nlp(doc)
    for ent in text.ents:
        print(f"{ent.text:<15} Label: {ent.label_:<8} ({spacy.explain(ent.label_)})")


In [50]:
itemsBoughtFromNouns(doc)

Items Bought: ['gas', 'tank']


In [ ]:
for doc in df["text"]:
    itemsBoughtFromNouns(doc)   

I went to the mall today and bought a T-shirt for $10  - Items Bought: ['mall', 'today']
Paid 150 EGP for an Uber ride to Cairo Festival City  - Items Bought: ['ride']
Grabbed a large iced latte at Starbucks for $5.75 this morning  - Items Bought: ['latte', 'morning']
Ordered a pepperoni pizza from Papa Johns for 320 EGP  - Items Bought: ['pepperoni', 'pizza']
Bought wireless noise cancelling headphones on Amazon for $199.99  - Items Bought: ['noise', 'headphones']
Spent 850 pounds on groceries at Carrefour yesterday  - Items Bought: ['pounds', 'groceries']
Filled up the gas tank at Mobil for 450 EGP  - Items Bought: ['gas', 'tank']
Purchased a pair of running shoes from Nike for $120  - Items Bought: ['pair', 'shoes']
Paid $14.99 for my monthly Netflix subscription  - Items Bought: ['subscription']
Bought two cinema tickets at VOX Cinemas for 360 EGP  - Items Bought: ['cinema', 'tickets']
Got a haircut at the barbershop for 120 EGP  - Items Bought: ['haircut', 'barbershop']
Bought a p

In [54]:
import random
import spacy
import pandas as pd
from spacy.training import Example

df = pd.read_csv("../data/expense_ner.csv").fillna("")
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

labelColumns = {
    "item": "ITEM",
    "merchant": "MERCHANT",
    "date_ref": "DATE"
}

for label in labelColumns.values():
    ner.add_label(label)

trainingData = []
for _, row in df.iterrows():
    text = row["text"]
    entities = []
    for col, label in labelColumns.items():
        val = str(row[col]).strip()
        if val:
            start = text.find(val)
            if start != -1:
                end = start + len(val)
                overlap = any(not (end <= s or start >= e) for s, e, _ in entities)
                if not overlap:
                    entities.append((start, end, label))
    trainingData.append((text, {"entities": entities}))

random.shuffle(trainingData)
trainSplit = trainingData[:160]
testSplit = trainingData[160:]

optimizer = nlp.begin_training()
for epoch in range(25):
    random.shuffle(trainSplit)
    losses = {}
    for text, annotations in trainSplit:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.2, sgd=optimizer, losses=losses)

for text, _ in testSplit[:10]:
    doc = nlp(text)
    extracted = [(ent.text, ent.label_) for ent in doc.ents]
    print("", text, "->", extracted)

 Bought coffee beans at LC Waikiki for $243.4 -> [('coffee beans', 'ITEM'), ('LC Waikiki', 'MERCHANT')]
 Spent 1400.0 EGP on cat treats at Best Buy on Monday -> [('cat treats', 'ITEM'), ('Best Buy', 'MERCHANT'), ('on Monday', 'DATE')]
 Bought a hoodie at Target for $210.53 -> [('hoodie', 'ITEM'), ('Target', 'MERCHANT')]
 Paid 1990.0 EGP for a power bank via InstaPay today -> [('power bank', 'ITEM'), ('InstaPay', 'MERCHANT'), ('today', 'DATE')]
 Picked up sunglasses from H&M and paid 2960.0 EGP -> [('sunglasses', 'ITEM'), ('H&M', 'MERCHANT')]
 Ordered iced americano from H&M for 1300.0 EGP -> [('iced americano', 'ITEM'), ('H&M', 'MERCHANT')]
 Paid 3450.0 EGP for cat treats via InstaPay today -> [('cat treats', 'ITEM'), ('InstaPay', 'MERCHANT'), ('today', 'DATE')]
 Grabbed a laptop stand at LC Waikiki for $88.89 on Monday -> [('laptop stand', 'ITEM'), ('LC Waikiki', 'MERCHANT'), ('on Monday', 'DATE')]
 Grabbed sneakers at Carrefour for $239.51 -> [('sneakers', 'ITEM'), ('Carrefour', 'MER

In [62]:
doc = nlp("I got a bike")
extracted = [(ent.text, ent.label_) for ent in doc.ents]
print("", text, "->", extracted)

 Purchased a domain name on Namecheap for $11.28 -> [('bike', 'ITEM')]
